Quiz 51 — Salary Outlier Detection  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

np.random.seed(140)
dept     = np.random.choice(["Engineering","Sales","Operations"], 80)
salaries = np.random.normal(45000, 12000, 80)
salaries[5]=180000; salaries[20]=2000; salaries[45]=200000; salaries[60]=1500

# ── 1. Z-score method ─────────────────────────────────────────────────────────
z           = (salaries - salaries.mean()) / salaries.std()
z_outliers  = np.where(np.abs(z) > 2.5)[0]
print("Z-score outliers  :", z_outliers)

# ── 2. IQR method ────────────────────────────────────────────────────────────
q1, q3  = np.percentile(salaries, [25, 75])
iqr     = q3 - q1
lower   = q1 - 1.5 * iqr
upper   = q3 + 1.5 * iqr
iqr_outliers = np.where((salaries < lower) | (salaries > upper))[0]
print("IQR outliers      :", iqr_outliers)

# ── 3. Remove union ───────────────────────────────────────────────────────────
all_outlier_idx = np.union1d(z_outliers, iqr_outliers)
print(f"\nBefore: {len(salaries)}  After: {len(salaries) - len(all_outlier_idx)}")

clean_sal   = np.delete(salaries, all_outlier_idx)
clean_dept  = np.delete(dept, all_outlier_idx)

# ── 4. Per-department stats (clean) ──────────────────────────────────────────
clean_median = np.median(clean_sal)
print("\nDepartment stats (clean):")
for d in ["Engineering","Sales","Operations"]:
    m = clean_dept == d
    s = clean_sal[m]
    print(f"  {d:13s}: n={m.sum():2d}  mean=£{s.mean():,.0f}  "
          f"median=£{np.median(s):,.0f}  std=£{s.std():,.0f}")
    above_pct = (s > clean_median).mean() * 100
    print(f"             above company median: {above_pct:.1f}%")

# ── 5. Boxplot + stripplot ────────────────────────────────────────────────────
df = pd.DataFrame({"Department": clean_dept, "Salary": clean_sal})
plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x="Department", y="Salary", palette="pastel")
sns.stripplot(data=df, x="Department", y="Salary", color="navy", alpha=0.4, size=4)
plt.title("Clean Salary Distribution by Department")
plt.ylabel("Annual Salary (£)")
plt.tight_layout()
plt.savefig("hard_11_salary_outlier.png", dpi=100)
plt.show()

# ── WHY ───────────────────────────────────────────────────────────────────────
# Z-score is mean-based — sensitive to outliers in the mean/std calculation.
# IQR is median-based — resistant to outliers, often flags more conservatively.
# Using the union of both methods catches outliers that one method might miss.
# np.union1d returns sorted, deduplicated indices.